# Documentação

**Tabuleiro**

Determinado por índices de 1 ... 64 em sistema linear

```
1  2  3  4  5  6  7  8
9 10 11 12 13 14 15 16
17 ...
...
57 58 59 60 61 62 63 64
```

Determinado por índices de (0, 0) ... (7, 7) em sistemas de coordenadas
```
(0, 0) (1, 0) (2, 0) (3, 0) (4, 0) (5, 0) (6, 0) (7, 0)
(0, 1) (1, 1) (2, 1) (3, 1) (4, 1) (5, 1) (6 ,1) (7, 1)
(0, 2) (1, 2) ...
...
(0, 7) (1, 7) (2, 7) (3, 7) (4, 7) (5, 7) (6, 7) (7, 7)
```

---

``` tabuleiro[y][x] = id + 1 ``` armazena o id da peça + 1 na casa (x, y), 0 se vazia

**Peças**

Cada peça possui um id que vai de 0 ... 15 para as pretas e 16 .. 31 para brancas

O valor ``` pecas[i] = pos ``` representa a posição linear da peça de id ``` i ```\
Se a peça foi capturada, sua posição é 0 (fora do tabuleiro)

---

Conversão de posições lineares p/ coordenadas

```
x = (pieces[i]-1) % 8
y = (pecas[i]-1) // 8
```

Coluna: Fazemos um (mod 8) para iterar pelas colunas, já que cada linha possui 8 casas

Linha: Divisão inteira (```//```) por 8 pois dá exatamente o valor dde quantas linhas já passaram

Obs: ``` pieces[i]-1 ``` para converter a posição de índices que vão de 1 .. 64 para 0 ... 63. Fazemos isso para poder trabalhar com divisão por 8


# Imports

In [ ]:
from abc import ABC, abstractmethod
import time

# Variáveis globais

In [ ]:
pecas = []
valor = []
tabuleiro = []
curr_player = 1

# Código Sem POO

## Movimento das Peças

In [ ]:
def get_cor(id_):
  """
    False (0) se preta
    True (1) se branca

    parâmetros:
      id (int): id da peça
    retorna:
      cor (bool): cor da peça
  """
  return id_ > 15

def linear_para_coord(id_):
    """
      Transforma uma posição linear em coordenadas

      parâmetros:
        id (int): posição linear
      retorna:
        (x, y): posição em coordenadas
    """
    x = (pecas[id_]-1) % 8
    y = (pecas[id_]-1) // 8
    return (x, y)

def get_mat_add(id_):
  """
    Retorna o valor de captura de uma peça
  """
  return valor[id_]*(2*curr_player - 1)

In [ ]:
def torre_casas_cobertas(x, y):
  """
    Gera todos as casas cobertas pela torre a partir de
      uma posição (x, y)

    par^amêtros:
      x (int): coluna
      y (int): linha
    retorna:
      mov (list): lista de movimentos
  """
  y_cima = range(y-1, -1, -1)
  cima = [y_cima, [x]]

  y_baixo = range(y+1, 8)
  baixo = [y_baixo, [x]]

  x_esq = range(x-1, -1, -1)
  esq = [[y], x_esq]

  x_dir = range(x+1, 8)
  dir_ = [[y], x_dir]

  return [cima, baixo, esq, dir_]


def torre_mov(movimentos):
  """
    Gera todos os possíveis movimentos para as torres do jogador atual.
    Consideramos a dama também pq ela anda que nem uma torre

    parâmetros:
      mov (ref lista): lista de movimentos, normalmente vazia e
        passada por referência
  """
  # Torres e Dama
  torres_pretas = [0, 7, 3]
  torres_brancas = [24, 31, 27]

  # Peças do jogador atual
  torres = [torres_pretas, torres_brancas][curr_player]

  for i in torres:
    if pecas[i] == 0:
      continue # Já capturada

    i_cor = get_cor(i)
    x, y = linear_para_coord(i)

    for range_y, range_x in torre_casas_cobertas(x, y):
      # Passamos por todas as possíveis casas da torre
      # range_y = range(...) e range_x = [x] ou
      # range_y = [y]     e    range_x = range(...)

      for casa_y in range_y:
        for casa_x in range_x:
          casa_peca = tabuleiro[casa_y][casa_x]
          if casa_peca == 0:
            # Casa vazia - Pode avançar
            movimentos.append([0, y, x, casa_y, casa_x, i+1, 0])

          elif i_cor + (casa_peca > 16) == 1:
            # Capturamos a peça
            mat_add = get_mat_add(casa_peca)
            movimentos.append([mat_add, y, x, casa_y, casa_x, i+1, casa_peca])
            break

          else:
            break

        else:
          continue

        break

In [ ]:
def bispo_casas_cobertas(x, y):
  y_cima = range(y-1, -1, -1)
  y_baixo = range(y+1, 8)
  y_lista = [y_cima, y_baixo]

  x_esq = range(x-1, -1, -1)
  x_dir = range(x+1, 8)
  x_lista = [x_esq, x_dir]

  return [y_lista, x_lista]


def bispo_mov(movimentos):
  # Consideramos a dama como bispo
  bispos_pretos = [2, 5, 3]
  bispos_brancos = [26, 29, 27]

  bispos = [bispos_pretos, bispos_brancos][curr_player]

  for i in bispos:
    if pecas[i] == 0:
      continue

    i_cor = get_cor(i)
    x, y = linear_para_coord(i)

    casas_cobertas = bispo_casas_cobertas(x, y)

    for range_y in casas_cobertas[0]:
      for range_x in casas_cobertas[1]:
        eixo_minimo = min(len(range_y), len(range_x))
        passos = range(eixo_minimo)
        for p in passos:
          casa_y = range_y[p]
          casa_x = range_x[p]

          casa_peca = tabuleiro[casa_y][casa_x]

          if casa_peca == 0:
            movimentos.append([0, y, x, casa_y, casa_x, i+1, 0])

          elif i_cor + (casa_peca > 16) == 1:
            mat_add = get_mat_add(casa_peca)
            movimentos.append([mat_add, y, x, casa_y, casa_x, i+1, casa_peca])
            break

          else:
            break

In [ ]:
def cavalo_mov(movimentos):
  cavalos_pretos = [1, 6]
  cavalos_brancos = [25, 30]

  cavalos = [cavalos_pretos, cavalos_brancos][curr_player]

  for i in cavalos:
    if pecas[i] == 0:
      continue

    i_cor = get_cor(i)
    x, y = linear_para_coord(i)

    for dist_y in [1, 2]:
      for dir_y in [-1, 1]:
        casa_y = y + dist_y * dir_y

        if casa_y < 0 or casa_y > 7:
          continue

        for dir_x in [-1, 1]:
          casa_x = x + (3 - dist_y) * dir_x

          if casa_x < 0 or casa_x > 7:
            continue

          casa_peca = tabuleiro[casa_y][casa_x]

          if casa_peca == 0 or ((i_cor) + (casa_peca > 16) == 1):
            mat_add = get_mat_add(casa_peca)
            movimentos.append([mat_add, y, x, casa_y, casa_x, i+1, casa_peca])

In [ ]:
def peao_mov(movimentos):
  dir_ = 1 - curr_player*2

  peoes_pretos = range(8, 16)
  peoes_brancos = range(16, 24)

  peoes = [peoes_pretos, peoes_brancos][curr_player]

  for i in peoes:
    if pecas[i] == 0:
      continue

    i_cor = get_cor(i)
    x, y = linear_para_coord(i)

    if y == 0 or y == 7:
      continue

    if x > 0 and tabuleiro[y+dir_][x-1] != 0 and (i_cor + (tabuleiro[y+dir_][x-1] > 16)) == 1:
      mat_add = get_mat_add(tabuleiro[y+dir_][x-1])
      movimentos.append([mat_add, y, x, y+dir_, x-1, i+1, tabuleiro[y+dir_][x-1]])

    if x < 7 and tabuleiro[y+dir_][x+1] != 0 and (i_cor + (tabuleiro[y+dir_][x+1] > 16)) == 1:
      mat_add = get_mat_add(tabuleiro[y+dir_][x+1])
      movimentos.append([mat_add, y, x, y+dir_, x+1, i+1, tabuleiro[y+dir_][x+1]])

    if tabuleiro[y+dir_][x] == 0:
      movimentos.append([0, y, x, y+dir_, x, i+1, 0])

      if y == [1, 6][curr_player] and tabuleiro[y + dir_*2][x] == 0:
        movimentos.append([0, y, x, y+dir_*2, x, i+1, 0])

In [ ]:
def rei_casas_cobertas(x, y):
  cima = max(0, y-1)
  baixo = min(8, y+2)
  y_lista = range(cima, baixo)

  esq = max(0, x-1)
  dir_ = min(8, x+2)
  x_lista = range(esq, dir_)


  return [y_lista, x_lista]

def rei_mov(movimentos):
  rei = [4, 28][curr_player]

  i_cor = get_cor(rei)
  x, y = linear_para_coord(rei)

  casas_cobertas = rei_casas_cobertas(x, y)

  for casa_y in casas_cobertas[0]:
    for casa_x in casas_cobertas[1]:
      casa_peca = tabuleiro[casa_y][casa_x]
      if casa_peca == 0 or (i_cor + (casa_peca > 16) == 1):
        mat_add = get_mat_add(casa_peca)
        movimentos.append([mat_add, y, x, casa_y, casa_x, rei+1, casa_peca])

#Função de avaliação

In [ ]:
def avaliacao():
  global material, valor_casa, valor, curr_player

  controle = 0

  movimentos = []
  torre_mov(movimentos)
  bispo_mov(movimentos)
  cavalo_mov(movimentos)
  peao_mov(movimentos)
  rei_mov(movimentos)

  k = 0.5

  for mat_add, from_y, from_x, to_y, to_x, peca, capturada in movimentos:
    controle += valor_casa[to_y][to_x] ** (1/3)

    if abs(mat_add) > 1:
      if valor[peca] < abs(mat_add):
        controle += min(9,abs(mat_add))*2*k
      elif valor[peca] == abs(mat_add):
        controle += min(9,abs(mat_add))*1.5*k
      else:
        controle += min(9,abs(mat_add))*k

  controle_oponente = 0

  curr_player = 1 - curr_player

  movimentos = []
  torre_mov(movimentos)
  bispo_mov(movimentos)
  cavalo_mov(movimentos)
  peao_mov(movimentos)
  rei_mov(movimentos)

  for mat_add, from_y, from_x, to_y, to_x, peca, capturada in movimentos:
    controle_oponente += valor_casa[to_y][to_x] ** (1/3)

    if abs(mat_add) > 1:
      if valor[peca] < abs(mat_add):
        controle_oponente += min(9,abs(mat_add))*2*k
      elif valor[peca] == abs(mat_add):
        controle_oponente += min(9,abs(mat_add))*1.5*k
      else:
        controle_oponente += min(9,abs(mat_add))*k


  curr_player = 1 - curr_player

  return material + controle*(2*curr_player - 1)/20 - controle_oponente*(2*curr_player - 1)/20

#Algoritmo de busca

In [ ]:
def minimax(alpha = float("inf"), beta = -float("inf")):
  global tabuleiro, pecas, valor, max_prof, prof, curr_player, material, movimentos

  # repetição
  if prof > 0 and tuple(pecas) in posicoes:
    return 0

  # avaliacao - termina busca
  if prof == max_prof:
    return avaliacao()

  # encontra movimentos
  movimentos = []
  torre_mov(movimentos)
  bispo_mov(movimentos)
  cavalo_mov(movimentos)
  peao_mov(movimentos)
  rei_mov(movimentos)

  # ordenando por relevância
  if curr_player == 1:
    movimentos.sort(reverse=True, key=lambda x:x[0]+len(x)/100)
  else:
    movimentos.sort(key=lambda x:x[0]-len(x)/100)

  melhor = float("inf")*(1 - 2*curr_player)
  jogada = []

  prof += 1

  # buscando mais a fundo
  for mat_add, from_y, from_x, to_y, to_x, peca, capturada in movimentos:
    #xeque-mate encontrado
    if abs(mat_add) == float("inf"):
      melhor = mat_add
      movimento = [mat_add, from_y, from_x, to_y, to_x, peca, capturada]

      if curr_player == 1:
        beta = max(beta, mat_add)
      else:
        alpha = min(alpha, mat_add)

    if beta >= alpha:
      break

    # fazendo jogada
    tabuleiro[from_y][from_x] = 0
    tabuleiro[to_y][to_x] = peca

    pecas[peca-1] = to_y * 8 + to_x + 1
    if capturada != 0:
      pecas[capturada-1] = 0

    material += mat_add

    curr_player = 1 - curr_player

    aval_mov = minimax(alpha, beta)

    curr_player = 1 - curr_player

    if curr_player == 1:
      if aval_mov >= melhor:
        beta = max(beta, aval_mov)
        melhor = aval_mov
        movimento = [mat_add, from_y, from_x, to_y, to_x, peca, capturada]
    else:
      if aval_mov <= melhor:
        alpha = min(alpha, aval_mov)
        melhor = aval_mov
        movimento = [mat_add, from_y, from_x, to_y, to_x, peca, capturada]

    # desfazendo movimento
    tabuleiro[from_y][from_x] = peca
    tabuleiro[to_y][to_x] = capturada

    pecas[peca-1] = from_y * 8 + from_x + 1
    if capturada != 0:
      pecas[capturada-1] = to_y * 8 + to_x + 1

    material -= mat_add

  prof -= 1

  if prof != 0:
    return melhor

  return [*movimento, melhor]

#Estado inicial

In [ ]:
tabuleiro = [[i for i in range(1,9)],
         [i for i in range(9,17)],
         [0]*8,
         [0]*8,
         [0]*8,
         [0]*8,
         [i for i in range(17,25)],
         [i for i in range(25,33)]]

pecas = [*[i for i in range(1,17)], *[i for i in range(49,65)]]

valor = [0, 5, 3, 3, 9, float("inf"), 3, 3, 5,
            1, 1, 1, 1, 1, 1, 1, 1,
            1, 1, 1, 1, 1, 1, 1, 1,
            5, 3, 3, 9, float("inf"), 3, 3, 5]

valor_casa = [[0.50,0.52,0.55,0.57,0.57,0.55,0.52,0.50],
                [0.52,0.55,0.60,0.70,0.70,0.60,0.55,0.52],
                [0.55,0.60,0.70,0.80,0.80,0.70,0.60,0.55],
                [0.57,0.65,0.85,1.00,1.00,0.85,0.65,0.57],
                [0.57,0.65,0.85,1.00,1.00,0.85,0.65,0.57],
                [0.55,0.60,0.70,0.80,0.80,0.70,0.60,0.55],
                [0.52,0.55,0.60,0.70,0.70,0.60,0.55,0.52],
                [0.50,0.52,0.55,0.57,0.57,0.55,0.52,0.50]]

string = {0:"OO", 1:"BR", 2:"BN", 3:"BB", 4:"BQ", 5:"BK", 6:"BB", 7:"BN", 8:"BR",
                  9:"BP", 10:"BP", 11:"BP", 12:"BP", 13:"BP", 14:"BP", 15:"BP", 16:"BP",
                  17:"WP", 18:"WP", 19:"WP", 20:"WP", 21:"WP", 22:"WP", 23:"WP", 24:"WP",
                  25:"WR", 26:"WN", 27:"WB", 28:"WQ", 29:"WK", 30:"WB", 31:"WN", 32:"WR"}

curr_player = 1

prof = 0
max_prof = 5
material = 0

#Loop de jogo

In [ ]:
# jogando
posicoes = {tuple(pecas):1}
while True:
    time_ = time.time()
    mat_add, from_y, from_x, to_y, to_x, peca, capturada, melhor = minimax()
    print(time.time() - time_)

    tabuleiro[from_y][from_x] = 0
    tabuleiro[to_y][to_x] = peca

    pecas[peca-1] = to_y * 8 + to_x + 1
    if capturada != 0:
        pecas[capturada-1] = 0

    material += mat_add

    curr_player = 1 - curr_player

    if tuple(pecas) in posicoes:
        posicoes[tuple(pecas)] += 1

        if posicoes[tuple(pecas)] == 3:
            print("Tie")
            break

    else:
        posicoes[tuple(pecas)] = 1

    print(from_y+1, from_x+1, to_y+1, to_x+1, material, melhor)
    for x in tabuleiro:
        line = ""
        for peca in x:
            line += string[peca] + " "

        print(line)

    print("")

    input_ = input().strip()
    from_x = "abcdefgh".index(input_[0])
    from_y = 8-int(input_[1])
    to_x = "abcdefgh".index(input_[2])
    to_y = 8-int(input_[3])

    peca = tabuleiro[from_y][from_x]
    capturada = tabuleiro[to_y][to_x]
    mat_add = -valor[capturada]

    tabuleiro[from_y][from_x] = 0
    tabuleiro[to_y][to_x] = peca

    pecas[peca-1] = to_y * 8 + to_x + 1
    if capturada != 0:
        pecas[capturada-1] = 0

    material += mat_add

    curr_player = 1 - curr_player

    if tuple(pecas) in posicoes:
        posicoes[tuple(pecas)] += 1

        if posicoes[tuple(pecas)] == 3:
            print("Tie")
            break

    else:
        posicoes[tuple(pecas)] = 1

    for x in tabuleiro:
        line = ""
        for peca in x:
            line += string[peca] + " "

        print(line)

    print("")

    if material == float("inf"):
        print("Brancas vencem!")
        break

    elif material == -float("inf"):
        print("Pretas vencem!")
        break

7.342428207397461
7 5 6 5 0 1.2047723247070055
BR BN BB BQ BK BB BN BR 
BP BP BP BP BP BP BP BP 
OO OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO WP OO OO OO 
WP WP WP WP OO WP WP WP 
WR WN WB WQ WK WB WN WR 



KeyboardInterrupt: Interrupted by user